# 卷积神经网络练习（第 6 章）



- CNN 概述（卷积层、池化层、全连接层的作用）
- 卷积层：卷积运算、填充、步幅、3 维数据的卷积运算、`nn.Conv2d` 的使用
- 池化层：Max 池化与 Average 池化、`nn.MaxPool2d` / `nn.AvgPool2d`
- 应用案例：Fashion-MNIST 服装分类（加载数据、搭建模型、模型训练）

说明：本练习在教材示例的基础上做了适当调整（卷积核大小、通道数、激活函数、初始化方式、
优化器等均与教材不同），请先阅读题目描述，再在下方代码单元格中**手写代码**完成练习，
写完后与 `answer` 目录下的答案对照。

部分练习所需的数据位于项目根目录的 `data/` 下，本 notebook 中使用相对路径 `../../data/...`。

先执行下面的单元格导入所需的库。

In [ ]:
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.set_printoptions(precision=3, suppress=True)

print("torch version:", torch.__version__)

## 6.1 卷积层

卷积层对数据进行卷积运算：以一定间隔滑动卷积核的窗口，将各个位置上卷积核的元素和输入的
对应元素相乘再求和（乘积累加）。卷积核的参数相当于权重，此外还有偏置。

卷积层接收 3 维形状（通道, 高, 宽）的输入，同样以 3 维形状输出，因此不会像全连接层那样
丢失空间信息。

### 练习 1：手写单通道二维卷积运算

请实现一个不借助 `F.conv2d` 的卷积函数，并用它计算下面 5×5 输入与 3×3 卷积核的卷积结果。

要求：
1. 实现 `conv2d_naive(x, w, b=None, stride=1, padding=0)`，其中 `x` 形状为 `(C, H, W)`，
   `w` 形状为 `(FN, C, FH, FW)`，返回形状 `(FN, OH, OW)`；`padding` 用 0 填充；
2. 输入使用下面的 `x`（先 `unsqueeze` 成 `(1, 5, 5)`）、卷积核使用 `w`
   （先 `unsqueeze` 成 `(1, 1, 3, 3)`），步幅 1、无填充，打印卷积结果；
3. 用 `F.conv2d` 计算同样的卷积，验证你的结果与之相等（`torch.allclose`）。

In [ ]:
# 练习 1：手写单通道二维卷积运算
# TODO: 请在此处手写代码完成练习

### 练习 2：填充（padding）

填充是在输入数据周围填入固定数据（通常为 0），用于调整输出数据的形状大小。

请基于练习 1 的 `conv2d_naive`，对形状为 `(1, 4, 4)`、元素全为 1 的输入数据，
使用 3×3、元素全为 1 的卷积核，分别计算：

1. 不填充（`padding=0`）时的输出形状；
2. 填充幅度为 1（`padding=1`）时的输出形状。

打印两种情况的输出形状，并说明填充是如何影响输出形状的。

In [ ]:
# 练习 2：填充对输出形状的影响
# TODO: 请在此处手写代码完成练习

### 练习 3：步幅（stride）

步幅是应用卷积核的位置间隔。请基于练习 1 的 `conv2d_naive`，对形状为 `(1, 4, 4)`
的输入（元素全为 1）应用幅度为 1 的填充，并使用 3×3 的卷积核、**步幅为 3** 进行卷积：

1. 打印输出的形状（想一想为什么是 2×2）；
2. 用 `F.conv2d` 验证结果一致。

In [ ]:
# 练习 3：步幅的卷积运算
# TODO: 请在此处手写代码完成练习

### 练习 4：输出尺寸计算公式

假设输入为 `(H, W)`，卷积核为 `(FH, FW)`，填充为 `P`，步幅为 `S`，则输出尺寸为：

$$OH=\frac{H+2P-FH}{S}+1$$

请实现函数 `conv_output_size(H, FH, P, S)`（结果为向下取整，与 PyTorch 保持一致），
并计算下面几组参数对应的输出边长：

| H | FH | P | S |
|---|----|---|---|
| 4 | 3 | 1 | 3 |
| 28 | 5 | 2 | 1 |
| 28 | 3 | 1 | 2 |
| 7 | 3 | 0 | 1 |

In [ ]:
# 练习 4：实现输出尺寸计算公式
# TODO: 请在此处手写代码完成练习

### 练习 5：3 维数据（多通道）的卷积运算

图像是 3 维数据（通道, 高, 宽）。在 3 维卷积中，输入数据的通道数与卷积核的通道数必须
相同；使用 `FN` 个卷积核就能得到 `FN` 个输出通道。

请基于练习 1 的 `conv2d_naive`：

1. 用 `torch.randn` 构造形状为 `(3, 5, 5)` 的输入 `x` 和形状为 `(2, 3, 3, 3)` 的卷积核 `w`、
   形状为 `(2,)` 的偏置 `b`（`torch.manual_seed(42)` 保证可复现）；
2. 计算卷积，打印输出形状（应为 `(2, 3, 3)`）；
3. 用 `F.conv2d`（输入需 `unsqueeze` 成 `(1, 3, 5, 5)`）验证结果一致。

In [ ]:
# 练习 5：3 维数据（多通道）的卷积运算
# TODO: 请在此处手写代码完成练习

### 练习 6：`nn.Conv2d` 的参数与输出形状

`nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)`：
`in_channels` 为输入通道数，`out_channels` 为输出通道数（卷积核个数），
`kernel_size` 为卷积核大小，`stride` 为步幅，`padding` 为填充幅度。

请构造卷积层 `nn.Conv2d(3, 16, kernel_size=5, stride=2, padding=2)`，
对形状为 `(1, 3, 32, 32)` 的输入做卷积：

1. 用练习 4 的公式计算输出的高和宽，再打印实际输出形状验证；
2. 计算该卷积层的参数总数（权重元素个数 + 偏置个数），并与
   `sum(p.numel() for p in conv.parameters())` 的结果比较。

In [ ]:
# 练习 6：Conv2d 参数与输出形状计算
# TODO: 请在此处手写代码完成练习

### 练习 7：用 `nn.Conv2d` 处理真实图片

请读取 `../../data/duck.jpg`，将其转换为张量并调整为 `(C, H, W)` 的形状，
然后使用 `nn.Conv2d(in_channels=3, out_channels=3, kernel_size=5, stride=4, padding=1, bias=False)`
进行卷积（注意教材用的是 9×9 卷积核、步幅 3，这里请按上面的参数实现）：

1. 打印输入、输出特征图的形状；
2. 将输出特征图转换回图片并可视化（原图与输出图并排显示）。

In [ ]:
# 练习 7：用 Conv2d 处理真实图片
# TODO: 请在此处手写代码完成练习

## 6.2 池化层

池化层通过缩小长、宽方向上的空间来降维，能够缩减模型大小、提高计算速度。常见的池化有
Max 池化（取窗口内最大值）和 Average 池化（取窗口内平均值）。

池化层**没有要学习的参数**，且池化运算**按通道独立进行**，经过池化后通道数不变。

### 练习 8：手写 Max 池化

请实现 `maxpool_naive(x, k=2, s=2)`，`x` 形状为 `(C, H, W)`，返回形状 `(C, OH, OW)`，
对每个通道独立地在 `k×k` 窗口内取最大值：

1. 用 `torch.manual_seed(0)` 构造形状为 `(1, 4, 4)` 的随机输入；
2. 用你的函数做 2×2、步幅 2 的 Max 池化，打印输出形状；
3. 用 `nn.MaxPool2d(kernel_size=2, stride=2)` 验证结果一致。

In [ ]:
# 练习 8：手写 Max 池化并与 nn.MaxPool2d 对比
# TODO: 请在此处手写代码完成练习

### 练习 9：手写 Average 池化

请实现 `avgpool_naive(x, k=2, s=2)`，对每个通道独立地在 `k×k` 窗口内取平均值：

1. 使用与练习 8 相同的随机输入（`(1, 4, 4)`，2×2、步幅 2）；
2. 打印 Average 池化的结果；
3. 用 `nn.AvgPool2d(kernel_size=2, stride=2)` 验证结果一致。

In [ ]:
# 练习 9：手写 Average 池化并与 nn.AvgPool2d 对比
# TODO: 请在此处手写代码完成练习

### 练习 10：池化的鲁棒性

池化的一个特点是对微小偏差具有鲁棒性：当数据发生微小偏差（如向宽度方向平移 1 个元素）时，
池化的输出可能保持不变。

请构造输入 `x`（形状 `(1, 4, 4)`，每一行都是 `[1, 9, 9, 9]`），并对它以及
"整体向左平移 1 个元素、末尾补 0 后"的数据 `x_shift` 分别做 2×2、步幅 2 的 Max 池化，
比较两次的输出是否相同，并打印结果。

提示：平移可用 `torch.cat([x[:, :, 1:], torch.zeros_like(x[:, :, :1])], dim=-1)`。

In [ ]:
# 练习 10：验证池化对微小偏差的鲁棒性
# TODO: 请在此处手写代码完成练习

## 6.3 应用案例：服装分类

Fashion-MNIST 数据集中每个样本都是 28×28 的灰度图像，对应 10 个类别
（0 T恤/上衣，1 裤子，2 套头衫，3 连衣裙，4 外套，5 凉鞋，6 衬衫，7 运动鞋，8 包，9 靴子）。

注意：教材使用的 `fashion-mnist_train.csv` / `fashion-mnist_test.csv` 在本地 `data/` 下不可用，
本练习改用 `../../data/train.csv`（格式相同：第 1 列为标签，第 2~785 列为 784 个像素），
并自行按 8:2 划分训练集与测试集。

### 练习 11：加载数据并划分训练集/测试集

请读取 `../../data/train.csv`，完成以下步骤：

1. 将像素列转换为浮点张量并 reshape 成 `(N, 1, 28, 28)`，将标签列转换为 `torch.int64`；
2. 使用 `torch.randperm` 按 8:2 划分训练集与测试集（注意训练/测试的划分要基于同一个随机排列）；
3. 用 `TensorDataset` 封装为 `train_dataset`、`test_dataset`，打印各自的样本数；
4. 可视化训练集中的某一个样本（`cmap="gray"`），并打印它的标签。

提示：像素值范围是 0~255，可以除以 255 归一化，以便训练更稳定。

In [ ]:
# 练习 11：加载数据并划分训练集/测试集
# TODO: 请在此处手写代码完成练习

### 练习 12：搭建卷积神经网络模型

请用 `nn.Sequential` 搭建如下结构的模型（与教材不同，这里使用 ReLU、Max 池化和 Dropout）：

| 层 | 说明 |
|----|------|
| Conv2d(1, 8, kernel_size=3, padding=1) | 输出 8 通道，尺寸不变 |
| ReLU | 激活 |
| MaxPool2d(2, 2) | 28 → 14 |
| Conv2d(8, 32, kernel_size=3, padding=1) | 输出 32 通道，尺寸不变 |
| ReLU | 激活 |
| MaxPool2d(2, 2) | 14 → 7 |
| Flatten | 拉平为 `32 × 7 × 7` |
| Linear(32 * 7 * 7, 128) | 全连接 |
| ReLU | 激活 |
| Dropout(0.3) | 随机失活 |
| Linear(128, 10) | 输出 10 个类别 |

搭建完成后，输入一个形状为 `(1, 1, 28, 28)` 的随机张量，逐层打印输出的形状，
确认与预期一致。

In [ ]:
# 练习 12：搭建卷积神经网络模型
# TODO: 请在此处手写代码完成练习

### 练习 13：编写模型训练函数

请实现 `train(model, train_dataset, test_dataset, lr, epoch_num, batch_size, device)`：

- 对卷积层和全连接层的权重使用 **He（Kaiming）正态初始化**
  （`nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")`）；
- 使用交叉熵损失 `nn.CrossEntropyLoss` 和 **Adam** 优化器（教材用的是 SGD）；
- 每个 epoch：在训练集上训练，记录**平均训练损失**和**训练准确率**；
  再在测试集上评估，记录**测试准确率**（评估时用 `torch.no_grad()`）；
- 返回三个列表：`train_loss_list`、`train_acc_list`、`test_acc_list`。

提示：`kaiming_normal_` 会直接修改权重张量，但对 `nn.Sequential` 中的层用 `model.apply`
遍历时，`nn.Flatten`、`nn.ReLU` 等没有 `weight`，需要先判断类型。

In [ ]:
# 练习 13：模型训练函数
# TODO: 请在此处手写代码完成练习

### 练习 14：训练模型并绘制曲线

请调用练习 13 的 `train` 函数训练练习 12 的模型：

- `device` 使用 `torch.device("cuda" if torch.cuda.is_available() else "cpu")`；
- 学习率 `lr=0.001`，`epoch_num=3`，`batch_size=128`；
- 将训练损失、训练准确率、测试准确率画在同一张图上
  （损失用左轴，准确率用右轴，或画成两张子图均可），并加上图例。

In [ ]:
# 练习 14：训练模型并绘制曲线
# TODO: 请在此处手写代码完成练习

## 6.4 深度卷积神经网络（思考题）

将网络层数加深可以更有效地提取层次信息。著名的深度卷积神经网络包括：

- **AlexNet**（2012）：5 个卷积层 + 3 个全连接层，使用 ReLU 与 Dropout；
- **VGG**（2014）：由多个卷积-池化层堆叠而成，常见 VGG-16 / VGG-19；
- **GoogleNet**（2014）：引入 Inception 结构，横向上使用多个不同大小的滤波器再合并；
- **ResNet**（2015）：引入"快捷结构"（残差连接），学习目标由 $h(x)$ 变为 $h(x)-x$，
  有效缓解深度网络的梯度消失问题。

### 练习 15：残差连接（思考 + 编码验证）

ResNet 的核心是残差连接：某一层的输出为 $y = h(x) + x$。

请编写代码验证：对于一个使用残差连接的简单模块，即使内部变换 $h$ 的权重很小，
输出相对于输入的梯度也不会消失。

要求：
1. 定义一个包含 `nn.Linear(8, 8)`（权重初始化为很小的值，如乘以 0.01）的残差模块，
   前向为 `x + self.fc(x)`；
2. 分别用"带残差"和"不带残差"（直接输出 `self.fc(x)`）两种方式，
   对输入 `x` 求 `y.sum()` 关于 `x` 的梯度；
3. 比较两种情况下梯度的范数，观察残差连接对梯度的影响。

In [ ]:
# 练习 15：思考题：残差连接与梯度传播
# TODO: 请在此处手写代码完成练习